# Lección 5.6 — Naive Bayes: probabilidad con supuestos ingenuos

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.1 (tipos de datos: texto), Lección 5.2 (probabilidades y umbral)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con frecuencias (sin fórmulas difíciles) cómo se actualiza una probabilidad al ver una **pista**.
- Describir cómo Naive Bayes **combina muchas pistas** y por qué su supuesto es "ingenuo".
- Usarlo para **texto** (`MultinomialNB`) y para datos numéricos (`GaussianNB`).
- Predecir el efecto de sus hiperparámetros (`alpha`, `var_smoothing`) y de las características repetidas.
- Reconocer cuándo es una buena elección y por qué sus probabilidades no son de fiar.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_wine, make_classification

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el detective que suma pistas

Un detective llega a la escena. **Antes** de ver ninguna pista, ya tiene una idea: "en este barrio, 3 de cada 10 robos los comete la banda X". Luego encuentra pistas:
- 👣 Huellas de botas → "la banda X usa botas" → **sube** la sospecha.
- 🧤 No hay huellas dactilares → "la banda X siempre usa guantes" → **sube** más.
- 🕐 Fue a mediodía → "la banda X solo actúa de noche" → **baja** la sospecha.

Cada pista **empuja** la sospecha hacia arriba o hacia abajo. Al final, el detective compara: ¿qué explica mejor **todas** las pistas juntas?

**Naive Bayes hace exactamente esto.** Empieza con lo que sabe antes de mirar (la proporción de cada clase) y cada característica es una pista que actualiza la probabilidad. El ejemplo más famoso: los **filtros de spam**, donde cada palabra del correo es una pista.

### ✋ Ejercicio 1 — Antes de mirar
En tu bandeja de entrada, 30 de cada 100 correos son spam. Llega un correo y **todavía no lo abres**.
1. ¿Qué probabilidad le das de ser spam?
2. Lo abres y dice "¡GRATIS!". ¿Tu probabilidad sube o baja? ¿Por qué?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **30 %**. Sin ninguna pista, lo mejor es la proporción general. A esto se le llama **probabilidad previa** (*prior*).
2. **Sube**, porque "gratis" aparece mucho más en el spam que en los correos normales. Cuánto sube depende de **qué tan distinta** es la frecuencia de esa palabra en cada clase. Lo calcularemos enseguida.
</details>

---
## 2. 📖 El concepto: actualizar con una pista (la regla de Bayes, con frecuencias)

La regla de Bayes suena intimidante, pero con **frecuencias** es solo contar. Imagina **1,000 correos**:

```
                       1,000 correos
                  ┌──────────┴───────────┐
              300 spam              700 normales
                  │                      │
  "gratis" en el 60 % de los spam    "gratis" en el 5 % de los normales
                  │                      │
          180 con "gratis"         35 con "gratis"
```

Si un correo dice "gratis", está entre esos **180 + 35 = 215** correos. ¿Cuántos de ellos son spam?

| Término | Qué es | En el ejemplo |
|---|---|---|
| **Probabilidad previa** (*prior*) | Lo que crees antes de ver la pista | 30 % spam |
| **Verosimilitud** (*likelihood*) | Qué tan frecuente es la pista en cada clase | 60 % en spam, 5 % en normales |
| **Probabilidad posterior** (*posterior*) | Lo que crees después de ver la pista | lo que vas a calcular |

### ✋ Ejercicio 2 — Calcula a mano
1. De los 215 correos con "gratis", ¿qué porcentaje es spam?
2. 🔮 ¿Y si en tu empresa solo 1 de cada 100 correos fuera spam (con los mismos 60 % y 5 %)? ¿Subiría o bajaría el resultado?

✍️ *Mis cálculos:* luego comprueba con la celda (✏️ cambia `PREVIA_SPAM` a 0.01).

In [ ]:
PREVIA_SPAM = 0.30        # ✏️ prueba 0.01
P_GRATIS_SI_SPAM, P_GRATIS_SI_NORMAL = 0.60, 0.05

spam_con_gratis = 1000 * PREVIA_SPAM * P_GRATIS_SI_SPAM
normal_con_gratis = 1000 * (1 - PREVIA_SPAM) * P_GRATIS_SI_NORMAL
print(f"De 1,000 correos: {spam_con_gratis:.0f} spam con 'gratis' y {normal_con_gratis:.0f} normales con 'gratis'")
print(f"Probabilidad de spam si dice 'gratis': {spam_con_gratis / (spam_con_gratis + normal_con_gratis):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. 180 / 215 ≈ **84 %**. Una sola palabra subió la sospecha de 30 % a 84 %.
2. **Baja muchísimo**: con 1 % de spam serían 6 spam y ≈50 normales con "gratis" → solo ≈11 %. La pista es la misma, pero la **probabilidad previa** importa: si el spam es rarísimo, incluso una pista fuerte no basta. (Es el mismo fenómeno de las pruebas médicas para enfermedades raras: un positivo no significa que estés enfermo.)
</details>

---
## 3. 📖 Cómo funciona: muchas pistas y un supuesto "ingenuo"

Con varias palabras, Naive Bayes hace algo muy simple: para cada clase, **multiplica** la previa por la verosimilitud de **cada** pista.

$$\text{puntaje(spam)} = P(\text{spam}) \times P(\text{pista}_1 \mid \text{spam}) \times P(\text{pista}_2 \mid \text{spam}) \times \dots$$

Hace lo mismo con "normal" y compara: los puntajes se reparten para que sumen 100 %.

**¿Por qué "ingenuo" (*naive*)?** Porque multiplicar así solo es correcto si las pistas son **independientes** entre sí dentro de cada clase. Y casi nunca lo son: "ganaste" y "premio" suelen aparecer **juntas**. Naive Bayes lo ignora y cuenta las dos como pistas separadas. Sorprendentemente, aun así suele **clasificar** bastante bien.

Su "entrenamiento" es solo **contar**: qué proporción de cada clase hay y con qué frecuencia aparece cada pista en cada clase. Nada de descenso de gradiente (Lección 3.3): por eso es rapidísimo.

Estas son las frecuencias que "aprendió" nuestro filtro:

In [ ]:
frecuencias = pd.DataFrame({
    "en spam":    [0.60, 0.40, 0.05],
    "en normal":  [0.05, 0.01, 0.30],
}, index=["gratis", "ganaste", "reunión"])
previa = {"en spam": 0.30, "en normal": 0.70}
frecuencias

### 🔮 Ejercicio 3 — Tira y afloja
Llega un correo con las palabras **"gratis"** y **"reunión"**. "Gratis" empuja hacia spam; "reunión", hacia normal. ¿Quién gana?

✍️ *Mi apuesta:*

In [ ]:
PALABRAS = ["gratis", "reunión"]        # ✏️ prueba ["gratis", "ganaste"] o ["reunión"]

puntajes = {}
for clase in ["en spam", "en normal"]:
    puntajes[clase] = previa[clase] * np.prod([frecuencias.loc[p, clase] for p in PALABRAS])
    print(f"Puntaje {clase:<9}: {previa[clase]} × " + " × ".join(str(frecuencias.loc[p, clase]) for p in PALABRAS)
          + f" = {puntajes[clase]:.4f}")
print(f"\nProbabilidad de spam: {puntajes['en spam'] / sum(puntajes.values()):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Casi empate, con ligera ventaja para **normal**! Probabilidad de spam ≈ **46 %**. "Gratis" multiplica la sospecha por 12 (0.60 / 0.05), pero "reunión" la divide entre 6 (0.30 / 0.05), y la previa favorece a "normal".

Con `["gratis", "ganaste"]` el spam gana por goleada (≈99 %): dos pistas fuertes en la misma dirección.
</details>

### 📖 El problema del cero
¿Qué pasa si una palabra **nunca** apareció en el spam de entrenamiento? Su frecuencia "en spam" sería 0, y como Naive Bayes **multiplica**, un solo cero anula todo lo demás.

### 🔮 Ejercicio 4 — Un cero lo arruina todo
Supón que en los 300 spam de entrenamiento la palabra "reunión" apareció **0 veces**. Llega este correo: *"¡GRATIS! ¡GANASTE! Reclama tu premio antes de la reunión"*. ¿Qué probabilidad de spam le dará el modelo?

✍️ *Mi predicción:*

In [ ]:
conteos = pd.DataFrame({"en spam": [180, 120, 0], "en normal": [35, 7, 210]},
                       index=["gratis", "ganaste", "reunión"])
totales = {"en spam": 300, "en normal": 700}
PALABRAS = ["gratis", "ganaste", "reunión"]

for ALPHA in [0, 1]:            # ✏️ prueba 0.1 y 10
    puntajes = {c: previa[c] * np.prod([(conteos.loc[p, c] + ALPHA) / (totales[c] + 2 * ALPHA) for p in PALABRAS])
                for c in totales}
    print(f"alpha={ALPHA}: probabilidad de spam = {puntajes['en spam'] / sum(puntajes.values()):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `alpha=0`: **0 %**. Un correo con "GRATIS" y "GANASTE" se declara 100 % normal solo porque contiene "reunión", una palabra que el modelo nunca vio en spam. ¡Absurdo!

La solución es el **suavizado** (*smoothing*, de Laplace): sumar un poquito (`alpha`) a todos los conteos, como si hubiéramos visto cada palabra al menos "un poquito" en cada clase. Con `alpha=1`, el mismo correo sale ≈ **66 %** spam: "reunión" sigue restando (casi nunca aparece en spam), pero ya no anula las otras dos pistas. `alpha` es el hiperparámetro principal de Naive Bayes para texto.
</details>

---
## 4. 📖 Naive Bayes con texto de verdad

Para usar texto, primero hay que convertirlo en números. La forma más simple es la **bolsa de palabras** (*bag of words*): una columna por cada palabra del vocabulario y, en cada fila, cuántas veces aparece. `CountVectorizer` lo hace por nosotros.

`MultinomialNB` es la versión de Naive Bayes pensada para **conteos** como estos.

In [ ]:
mensajes = [
    ("Ganaste un premio, reclama tu regalo gratis hoy", "spam"),
    ("Oferta exclusiva: gana dinero gratis desde casa", "spam"),
    ("Felicidades, fuiste seleccionado para un premio", "spam"),
    ("Últimas horas: descuento gratis, haz clic ya", "spam"),
    ("Tu cuenta fue bloqueada, haz clic para recuperar tu premio", "spam"),
    ("Gana un celular gratis respondiendo esta encuesta", "spam"),
    ("Crédito aprobado sin revisar buró, haz clic", "spam"),
    ("Promoción: dinero rápido y fácil, regalo seguro", "spam"),
    ("Reclama tu regalo antes de que expire la oferta", "spam"),
    ("Premio en efectivo esperándote, responde ya", "spam"),
    ("¿Nos vemos mañana en la reunión de las diez?", "normal"),
    ("Te mando el informe antes de la reunión", "normal"),
    ("Mamá, llego tarde a comer, guárdame algo", "normal"),
    ("¿Puedes revisar el documento que te envié?", "normal"),
    ("La reunión se cambió para el jueves", "normal"),
    ("Gracias por la cena de ayer, estuvo deliciosa", "normal"),
    ("¿A qué hora sale el vuelo mañana?", "normal"),
    ("Ya pagué la renta, te mando el comprobante", "normal"),
    ("El partido empieza a las ocho, ¿vienes?", "normal"),
    ("Recuerda comprar leche y pan de regreso", "normal"),
    ("Mañana no hay clase, el profesor está enfermo", "normal"),
    ("Te envío las fotos del viaje en la tarde", "normal"),
    ("¿Revisaste el presupuesto para la reunión?", "normal"),
    ("Feliz cumpleaños, que tengas un gran día", "normal"),
]
textos = [t for t, _ in mensajes]
etiquetas = [e for _, e in mensajes]

filtro = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0)).fit(textos, etiquetas)
vocabulario = filtro[0].get_feature_names_out()
print(f"{len(textos)} mensajes · vocabulario de {len(vocabulario)} palabras")
print("Algunas columnas:", list(vocabulario[:12]))

### 🔮 Ejercicio 5 — Ponle mensajes al filtro
Antes de ejecutar, predice qué dirá el filtro de cada mensaje:
1. *"Reclama tu premio gratis"*
2. *"¿Movemos la reunión al viernes?"*
3. *"Gratis la reunión de mañana"*
4. *"Hola"*

✍️ *Mis predicciones:*

In [ ]:
nuevos = [
    "Reclama tu premio gratis",
    "¿Movemos la reunión al viernes?",
    "Gratis la reunión de mañana",
    "Hola",
    # ✏️ escribe aquí tus propios mensajes
]
for texto, prob in zip(nuevos, filtro.predict_proba(nuevos)[:, list(filtro.classes_).index("spam")]):
    print(f"{prob:>5.0%} spam  ←  {texto}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Spam** (≈100 %): cuatro pistas de spam ("reclama", "tu", "premio", "gratis").
2. **Normal** (≈5 % spam): "reunión" solo aparece en mensajes normales.
3. **Normal** (≈3 % spam): "gratis" empuja hacia spam, pero "la", "reunión" y "mañana" empujan hacia normal (y la previa favorece a normal: hay 14 normales y 10 spam). Tres pistas le ganan a una.
4. ≈42 % = la **probabilidad previa** (10 de 24 mensajes son spam). "Hola" nunca apareció en el entrenamiento, así que no aporta ninguna pista: el modelo se queda con lo que sabía antes de mirar.
</details>

### ✋ Ejercicio 6 — ¿Qué palabras delatan al spam?
Naive Bayes es fácil de inspeccionar: basta comparar qué tan frecuente es cada palabra en cada clase. La celda muestra las palabras con mayor "poder de delación" (cuántas veces más probables son en spam que en normal).

In [ ]:
nb = filtro[-1]
log_prob = pd.DataFrame(nb.feature_log_prob_.T, index=vocabulario, columns=nb.classes_)
razon = np.exp(log_prob["spam"] - log_prob["normal"]).sort_values()
print("Más 'spam' (veces más probable en spam):")
print(razon.tail(6).iloc[::-1].round(1).to_string())
print("\nMás 'normal':")
print((1 / razon.head(6)).round(1).to_string())

<details><summary><b>👀 Ver respuesta</b></summary>

Las palabras más "spam" son **premio**, **tu**, **gratis**, **regalo**, **haz** y **clic**; las más "normales", **el**, **te**, **reunión**, **mañana**, **las** y **la**. ¡Lo descubrió solo contando!

Fíjate en que palabras sin significado como "tu", "el" o "la" también aparecen: con solo 24 mensajes, el modelo aprende cualquier diferencia de frecuencia, tenga sentido o no. Con más datos esas diferencias accidentales se diluyen.
</details>

---
## 5. 📖 Naive Bayes para números: `GaussianNB`

¿Y si las pistas no son palabras sino números (edad, temperatura, acidez)? `GaussianNB` supone que, **dentro de cada clase**, cada característica sigue una **campana** (distribución normal). Para entrenar solo calcula, por clase y por característica, el **promedio** y la **dispersión** de la campana.

Para predecir, mira en qué campana cae mejor cada valor y multiplica, igual que con las palabras.

| Variante | Para qué datos | Ejemplo |
|---|---|---|
| `MultinomialNB` | Conteos (palabras, eventos) | Clasificar textos |
| `BernoulliNB` | Sí/no (¿aparece la palabra?) | Textos cortos, casillas marcadas |
| `ComplementNB` | Conteos con clases desbalanceadas | Textos con categorías raras |
| `CategoricalNB` | Categorías | Color, ciudad… |
| `GaussianNB` | Números continuos | Mediciones, sensores |

Probémoslo con los 178 vinos de 3 productores (Lección 5.3).

In [ ]:
Xw, yw = load_wine(return_X_y=True, as_frame=True)

# Las 3 campanas de una característica: una por productor
for clase in [0, 1, 2]:
    Xw.loc[yw == clase, "flavanoids"].plot.kde(label=f"Productor {clase}")
plt.xlabel("flavanoids"); plt.legend(); plt.title("Una campana por clase"); plt.show()

### 🔮 Ejercicio 7 — El modelo más sencillo, ¿qué tan bueno?
Un `GaussianNB` sin escalar y sin ajustar nada. ¿Qué exactitud (validación cruzada) esperas comparado con el KNN escalado de la Lección 5.3 (≈95 %)?

✍️ *Mi apuesta:*

In [ ]:
print(f"GaussianNB             : {cross_val_score(GaussianNB(), Xw, yw, cv=5).mean():.1%}")
print(f"Regresión logística    : {cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), Xw, yw, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈ **97 %**, al nivel de la regresión logística escalada (≈98 %) y por encima del KNN, ¡sin escalar ni ajustar nada! Cuando las campanas de cada clase están bien separadas, como en la gráfica de `flavanoids`, Naive Bayes funciona muy bien.

> 💡 `GaussianNB` **no necesita escalar**: calcula una campana por característica, cada una en sus propias unidades.
</details>

---
## 6. 📖 ¿Qué pasa si las pistas no son independientes?

El supuesto ingenuo tiene un costo concreto: si dos pistas dicen **lo mismo**, Naive Bayes las cuenta **dos veces**. Es como si un testigo contara su historia tres veces y el detective creyera que son tres testigos distintos.

### 🔮 Ejercicio 8 — El testigo repetido
Tomamos un problema con 5 características y **copiamos 10 veces la más informativa**. Antes de ejecutar:
1. ¿Qué crees que pasa con la **exactitud**?
2. ¿Y con las **probabilidades** que da el modelo? ¿Habrá más o menos casos en los que diga estar "segurísimo" (más de 95 % o menos de 5 %)? Y cuando dice estar segurísimo, ¿acertará?

✍️ *Mis predicciones:*

In [ ]:
from sklearn.model_selection import cross_val_predict
X5, y5 = make_classification(n_samples=1000, n_features=5, n_informative=3, n_redundant=0, random_state=0)
COPIAS = 10     # ✏️ prueba 0, 3, 30
X_copias = np.hstack([X5] + [X5[:, [2]]] * COPIAS)      # la columna 2 es la más informativa

for nombre, X_ in [("Original", X5), (f"Con {COPIAS} copias", X_copias)]:
    prob = cross_val_predict(GaussianNB(), X_, y5, cv=5, method="predict_proba")[:, 1]
    seguro = (prob < 0.05) | (prob > 0.95)                 # casos donde dice estar "segurísimo"
    print(f"{nombre:<16} → exactitud {np.mean((prob > 0.5) == y5):.1%} · casos 'segurísimos': {seguro.mean():.0%} "
          f"· acierto en esos casos: {np.mean((prob[seguro] > 0.5) == y5[seguro]):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. La exactitud **baja** un poco (≈71 % → ≈69 %): la característica copiada ahora "grita" 11 veces y ahoga a las demás.
2. Lo grave son las **probabilidades**: los casos "segurísimos" pasan de ≈8 % a ≈88 %. Antes, cuando decía estar segurísimo, acertaba el 99 % de las veces; con las copias, solo el ≈72 %. El modelo se volvió **arrogante**: dice "95 %" cuando en realidad acierta 7 de cada 10.

**Lección práctica:** Naive Bayes suele **ordenar** bien los casos (quién es más sospechoso que quién), pero sus **probabilidades no son confiables** (están mal calibradas). Si vas a elegir un umbral con la fórmula de costos de la Lección 5.2, que supone probabilidades bien calibradas, desconfía de las de Naive Bayes: comprueba el umbral con validación cruzada.
</details>

---
## 7. 📖 Hiperparámetros: pocas perillas

| Hiperparámetro | Modelo | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|---|
| `alpha` | Multinomial, Bernoulli, Complement | Suavizado: cuánto "inventamos" para evitar ceros | Frecuencias más parecidas entre clases → modelo más tímido (**subajuste**) | Confía más en los conteos, incluso de palabras raras (**sobreajuste**) | 1.0; probar 0.01–10 |
| `var_smoothing` | Gaussian | Ensancha un poquito todas las campanas para estabilizar | Campanas más anchas, más tímido | Campanas exactas | 1e-9 (casi nunca hay que tocarlo) |
| `fit_prior` / `priors` | Todos | Si se usa la proporción real de clases como previa | — | — | `True`; fija `priors` si sabes que en producción la proporción será otra |

### ✏️ Ejercicio 9 — `alpha` en un filtro más grande
Generamos 600 reseñas de productos (positivas y negativas) y medimos la exactitud de `MultinomialNB` con distintos `alpha`.

In [ ]:
rng = np.random.default_rng(3)
positivas = ["excelente", "encantó", "rápido", "recomiendo", "perfecto", "bueno", "feliz", "calidad", "funciona", "bonito"]
negativas = ["malo", "roto", "tarde", "horrible", "devolví", "decepción", "caro", "nunca", "falla", "feo"]
neutras = ["producto", "envío", "caja", "precio", "llegó", "el", "la", "muy", "pero", "color", "talla",
           "día", "compré", "tienda", "para", "con", "una", "es", "fue", "mi"]

def generar_resenas(n):
    textos, sentimiento = [], []
    for _ in range(n):
        pos = rng.random() < 0.5
        propias, contrarias = (positivas, negativas) if pos else (negativas, positivas)
        palabras = list(rng.choice(neutras, rng.integers(5, 12)))
        palabras += list(rng.choice(propias, rng.integers(1, 3)))
        if rng.random() < 0.4:                                   # a veces mezclan ("bonito pero caro")
            palabras += list(rng.choice(contrarias, 1))
        rng.shuffle(palabras)
        textos.append(" ".join(palabras)); sentimiento.append(int(pos))
    return textos, np.array(sentimiento)

resenas, sentimiento = generar_resenas(600)
print("Ejemplos:", *[f"  {'👍' if s else '👎'} {t}" for t, s in zip(resenas[:3], sentimiento[:3])], sep="\n")
print()
for ALPHA in [0.001, 0.1, 1, 10, 100]:      # ✏️
    acc = cross_val_score(make_pipeline(CountVectorizer(), MultinomialNB(alpha=ALPHA)), resenas, sentimiento, cv=5).mean()
    print(f"alpha={ALPHA:<6} → exactitud CV {acc:.1%}  " + "█" * int((acc - 0.5) * 80))

¿Qué `alpha` elegirías? ¿Por qué un `alpha` enorme empeora?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

Valores de 0.001 a 10 dan resultados casi idénticos (≈89–90 %): con 600 reseñas y un vocabulario pequeño, casi no hay ceros que arreglar. Con `alpha=100` la exactitud cae a ≈78 %: el suavizado es tan fuerte que "inventa" 100 apariciones de cada palabra en cada clase y **borra las diferencias** entre clases (subajuste). Un `alpha` de 1 o menos es una elección razonable; confírmalo siempre con validación cruzada.
</details>

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Probabilidades de 0.00 o 1.00 en casi todo | Pistas correlacionadas o repetidas (supuesto ingenuo) | No fiarse de las probabilidades; calibrar (`CalibratedClassifierCV`); quitar características redundantes |
| Una palabra nueva o rara manda la probabilidad a 0 | `alpha` = 0 (sin suavizado) | `alpha` ≥ 0.1 |
| `ValueError: Negative values in data` | `MultinomialNB` con datos escalados (negativos) | No escalar los conteos; para números continuos, `GaussianNB` |
| Exactitud mediocre con datos numéricos | Las características no tienen forma de campana o hay interacciones fuertes | Transformar (logaritmos); probar otro modelo |
| Umbral calculado con la fórmula de costos funciona mal | Probabilidades mal calibradas | Elegir el umbral con `cross_val_predict` (Lección 5.2) |
| El modelo cambia mucho si cambian las proporciones de clases en producción | La previa se aprendió de los datos de entrenamiento | Fijar `priors` con la proporción esperada |

### 🔮 Ejercicio 10 — El error del que escala todo
Alguien, siguiendo la regla "siempre escala" de las lecciones 5.2–5.5, pone un `StandardScaler` antes de `MultinomialNB`. ¿Qué crees que pasará?

✍️ *Mi predicción:*

In [ ]:
X_conteos = CountVectorizer().fit_transform(resenas).toarray()
try:
    make_pipeline(StandardScaler(), MultinomialNB()).fit(X_conteos, sentimiento)
    print("Funcionó")
except ValueError as e:
    print("❌ ValueError:", e)

<details><summary><b>👀 Ver respuesta</b></summary>

**Error.** Al estandarizar, muchos valores se vuelven negativos, y `MultinomialNB` trabaja con **conteos** (que nunca son negativos). "Siempre escala" es una regla para los modelos basados en distancias o en coeficientes (KNN, SVM, logística), **no** para Naive Bayes ni para los árboles. Cada modelo tiene sus propias necesidades de preparación.
</details>

---
## 9. 📋 Ficha del modelo

| | Naive Bayes |
|---|---|
| **Tipo** | Supervisado: clasificación |
| **Intuición** | Un detective que parte de lo que sabe antes de mirar y actualiza su sospecha con cada pista, como si las pistas fueran independientes |
| **Cómo aprende** | Solo **cuenta**: la proporción de cada clase y la frecuencia (o la campana) de cada característica en cada clase |
| **Hiperparámetros clave** | `alpha` (suavizado, para texto), `var_smoothing` (Gaussian), `priors` |
| **✅ Fortalezas** | Rapidísimo para entrenar y predecir · Funciona con **pocos datos** · Excelente con texto y muchas características · Simple e inspeccionable · Buena línea base |
| **❌ Debilidades** | El supuesto de independencia casi nunca se cumple · **Probabilidades mal calibradas** (demasiado extremas) · No capta interacciones entre características · Suele perder contra modelos más flexibles con muchos datos |
| **Úsalo cuando…** | Clasificas texto (spam, temas, sentimiento), tienes pocos datos, necesitas algo rapidísimo o quieres una primera línea base |
| **Evítalo cuando…** | Necesitas probabilidades confiables para decidir con costos, o las interacciones entre variables son la clave del problema |

## 10. 🧠 Resumen en 5 puntos
1. Naive Bayes parte de la **probabilidad previa** de cada clase y la actualiza con cada **pista** (característica).
2. Multiplica las pistas como si fueran **independientes**: ese es su supuesto "ingenuo".
3. Entrenar es solo **contar**: por eso es rapidísimo y funciona con pocos datos.
4. El **suavizado** (`alpha`) evita que una pista nunca vista anule todo; `MultinomialNB` es para conteos (texto) y `GaussianNB` para números.
5. Suele **clasificar** bien, pero sus **probabilidades son exageradas**: no las uses tal cual para fijar umbrales por costos.

---
## 11. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 11.1 Repaso rápido (opción múltiple)

**1.** ¿Por qué se llama "ingenuo" (*naive*)?
- a) Porque no necesita datos · b) Porque supone que las características son independientes dentro de cada clase · c) Porque solo funciona con texto · d) Porque no tiene hiperparámetros

**2.** Un filtro de spam con `alpha=0` ve la palabra "jueves", que nunca apareció en spam. ¿Qué pasa?
- a) Nada · b) El correo tendrá 0 % de spam aunque tenga otras pistas fuertes · c) Da error · d) La probabilidad sube

**3.** ¿Qué hace realmente `GaussianNB.fit()`?
- a) Descenso de gradiente · b) Guarda todos los datos · c) Calcula, por clase, el promedio y la dispersión de cada característica · d) Construye un árbol

**4.** Tu `GaussianNB` da 0.999 o 0.001 en casi todos los casos. ¿Qué conclusión es **más** prudente?
- a) Es un modelo excelente y muy seguro · b) Las probabilidades probablemente están mal calibradas; no las uses tal cual para elegir umbrales · c) Hay que subir `alpha` · d) Hay que escalar los datos

**5.** ¿En cuál de estos casos Naive Bayes es una opción **especialmente** buena?
- a) 200 correos etiquetados para clasificar en 4 temas · b) Predecir el precio de una casa · c) Un problema donde lo importante es la interacción entre dos variables · d) Necesitas probabilidades perfectamente calibradas

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Multiplica las pistas como si no tuvieran relación entre sí.
2. **b)** Un cero en una multiplicación anula todo. Por eso se usa `alpha` > 0.
3. **c)** Solo cuenta y promedia: una campana por clase y característica.
4. **b)** Naive Bayes es famoso por sus probabilidades exageradas. (c no aplica: `alpha` no existe en `GaussianNB`.)
5. **a)** Texto + pocos datos es su terreno ideal. b) es regresión; c) y d) son justo sus debilidades.
</details>

### 11.2 Preguntas para pensar (sin ejecutar)

**A.** Una prueba de una enfermedad rara (1 de cada 1,000 personas) detecta al 99 % de los enfermos y da falso positivo al 2 % de los sanos. Te sale **positiva**. Usando frecuencias (imagina 100,000 personas), ¿qué probabilidad hay de que estés enfermo? ¿Qué tiene que ver con el Ejercicio 2?

**B. Caso de análisis 💼** En un banco, un analista compara modelos para detectar impagos con validación cruzada. Naive Bayes tiene la **peor exactitud** pero el **mejor *recall*** de los morosos. Su jefe dice: *"Usemos Naive Bayes y, para afinar, apliquemos la fórmula del umbral de la Lección 5.2 a sus probabilidades"*. ¿Qué le advertirías? ¿Qué harías en su lugar?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** De 100,000 personas: 100 enfermas → 99 positivos; 99,900 sanas → ≈ 1,998 falsos positivos. De ≈ 2,097 positivos, solo 99 están enfermos → **≈ 5 %**. Es el mismo efecto de la **probabilidad previa** del Ejercicio 2: si algo es rarísimo, incluso una pista fuerte deja la probabilidad baja. (Por eso a un positivo de una prueba de tamizaje le sigue una segunda prueba.)

**B.** Que el buen *recall* de Naive Bayes puede venir de que sus probabilidades **exageradas** cruzan el umbral de 0.5 con facilidad, no de que "entienda" mejor el riesgo. La fórmula $C_{FP}/(C_{FP}+C_{FN})$ supone probabilidades **calibradas**, y las de Naive Bayes no lo están (Sección 6). En su lugar: elegir el umbral **probando** varios con `cross_val_predict(..., method="predict_proba")` y midiendo el costo real, o calibrar primero las probabilidades; y comparar contra una regresión logística con el umbral ajustado, que suele estar mejor calibrada.
</details>

### 11.3 🧩 Práctica integradora: "Clasificador de reseñas con pocos datos"

Una tienda en línea quiere separar automáticamente las reseñas positivas de las negativas. El problema: el equipo solo tiene tiempo de etiquetar **unas pocas** a mano. ¿Qué modelo conviene cuando hay pocos datos?

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos.** Usamos el mismo generador de reseñas del Ejercicio 9. Generamos 2,000 reseñas y dejamos 500 para la prueba final.

In [ ]:
todas, sent_todas = generar_resenas(2_000)
r_tr, r_te, s_tr, s_te = train_test_split(todas, sent_todas, test_size=500, stratify=sent_todas, random_state=0)
print(f"Entrenamiento disponible: {len(r_tr)} · Prueba: {len(r_te)} · Positivas: {s_tr.mean():.0%}")

**Paso 2 🔮 — Con TODOS los datos de entrenamiento.** Comparamos Naive Bayes contra una regresión logística (con validación cruzada). ¿Quién crees que gana con 1,500 reseñas?

✍️ *Mi apuesta:*

In [ ]:
modelos = {
    "Naive Bayes": make_pipeline(CountVectorizer(), MultinomialNB()),
    "Logística":   make_pipeline(CountVectorizer(), LogisticRegression(max_iter=1000)),
}
for nombre, m in modelos.items():
    print(f"{nombre:<12} → exactitud CV {cross_val_score(m, r_tr, s_tr, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con 1,500 reseñas, los dos rondan el ≈90 % y prácticamente empatan (Naive Bayes ≈90.6 %, logística ≈89.8 %). Con suficientes datos, el supuesto ingenuo no estorba mucho en este problema.
</details>

**Paso 3 🔮 — Con POCOS datos.** Ahora entrenamos con solo **20, 50, 100, 300 y 1,500** reseñas y evaluamos en la prueba. Antes de ejecutar: ¿quién crees que aguanta mejor la escasez?

✍️ *Mi apuesta:*

In [ ]:
TAMANOS = [20, 50, 100, 300, 1_500]      # ✏️
resultados = {nombre: [] for nombre in modelos}
for n in TAMANOS:
    for nombre, m in modelos.items():
        resultados[nombre].append(m.fit(r_tr[:n], s_tr[:n]).score(r_te, s_te))

for nombre, accs in resultados.items():
    plt.plot(TAMANOS, accs, "o-", label=nombre)
plt.xscale("log"); plt.xlabel("Reseñas de entrenamiento (escala log)"); plt.ylabel("Exactitud en prueba")
plt.legend(); plt.grid(alpha=.3); plt.title("¿Quién aprende más rápido?"); plt.show()
print(pd.DataFrame(resultados, index=TAMANOS).round(3))

<details><summary><b>👀 Ver respuesta</b></summary>

Con **muy pocos** datos, Naive Bayes va claramente por delante: con 20 reseñas ≈65 % contra ≈58 %, y con 50, ≈73 % contra ≈65 %. Necesita estimar muy poco (frecuencias de palabras), mientras que la logística tiene que encontrar buenos coeficientes y con tan pocos ejemplos le cuesta más. Con 100–300 reseñas la logística ya la alcanza y la supera un poco (≈86 % y ≈90 %), y con 1,500 empatan (≈90 %). **Si tienes muy pocos datos etiquetados, Naive Bayes es una gran primera opción.**

> 📝 Aquí miramos la prueba varias veces solo para **ilustrar** una curva de aprendizaje (Lección 3.7), no para elegir un modelo. Para elegir, se usa validación cruzada.
</details>

**Paso 4 ✋ — Inspecciona.** Entrena Naive Bayes con las 1,500 reseñas y mira las 8 palabras más "positivas" y más "negativas".

In [ ]:
nb_final = make_pipeline(CountVectorizer(), MultinomialNB()).fit(r_tr, s_tr)
vocab = nb_final[0].get_feature_names_out()
lp = nb_final[-1].feature_log_prob_
diferencia = pd.Series(lp[1] - lp[0], index=vocab).sort_values()
print("Más negativas:", list(diferencia.index[:8]))
print("Más positivas:", list(diferencia.index[-8:][::-1]))
print("Más neutras  :", list(diferencia.abs().sort_values().index[:8]))

<details><summary><b>👀 Ver respuesta</b></summary>

Las listas de palabras positivas y negativas que encuentra coinciden con las del generador (`excelente`, `recomiendo`… / `horrible`, `devolví`…), y las neutras (`tienda`, `caja`, `la`…) quedan con una diferencia cercana a cero: aparecen igual en ambas clases y no aportan pistas. Un modelo que puedes **auditar** en 3 líneas.
</details>

**Paso 5 ✏️ — Prueba tus propias reseñas.** Escribe reseñas (usando palabras del vocabulario) y mira la probabilidad de "positiva". Intenta engañar al modelo.

In [ ]:
mis_resenas = [
    "excelente producto lo recomiendo",
    "llegó roto y tarde horrible",
    "bonito pero caro",
    "no es malo",            # 🤔
    # ✏️ escribe aquí las tuyas
]
for texto, p in zip(mis_resenas, nb_final.predict_proba(mis_resenas)[:, 1]):
    print(f"{p:>5.0%} positiva  ←  {texto}")

✍️ *¿Qué pasó con "no es malo"? ¿Por qué?*

<details><summary><b>👀 Ver respuesta</b></summary>

"no es malo" sale como **negativa**: la bolsa de palabras solo ve la palabra "malo" (una pista negativa fuerte) y no entiende que "no" la invierte. Además, "no" nunca apareció en el entrenamiento, así que el modelo la ignora. Naive Bayes (y cualquier modelo de bolsa de palabras) **no entiende el orden ni la negación**. Para eso hacen falta modelos que lean secuencias (Módulos 11–12).

"bonito pero caro" queda en ≈66 %: una pista positiva y una negativa que casi se compensan.
</details>

**Paso 6 ▶️ — Examen final.**

In [ ]:
print(f"Naive Bayes en prueba: {nb_final.score(r_te, s_te):.1%}")
print(f"Modelo 'siempre positiva': {max(s_te.mean(), 1 - s_te.mean()):.1%}")

✋ En dos frases, ¿qué le recomendarías a la tienda?

<details><summary><b>👀 Ver respuesta</b></summary>

Naive Bayes llega a ≈90 % en prueba (igual que en validación cruzada), contra ≈51 % de la línea base. Una recomendación: *"Empiecen con Naive Bayes: con solo unas decenas de reseñas etiquetadas ya funciona razonablemente bien, entrena en milisegundos y se puede revisar qué palabras usa. Cuando tengan más reseñas etiquetadas, compárenlo con una regresión logística y vigilen las frases con negaciones."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Naive Bayes, Clearly Explained!!!" y "Gaussian Naive Bayes" (inglés, con subtítulos).
- 📺 **Video:** 3Blue1Brown, "Bayes theorem, the geometry of changing beliefs" (con subtítulos en español).
- 📘 **Documentación:** scikit-learn, guía de usuario, secciones *Naive Bayes* y *Probability calibration*.

---
⬅️ **Anterior:** [5.5 Máquinas de vectores de soporte (SVM): la calle más ancha](5.5-maquinas-de-vectores-de-soporte.ipynb) · ➡️ **Siguiente:** [5.7 Comparativa: ¿qué modelo uso?](5.7-comparativa-que-modelo-uso.ipynb)